# Projekt Datenanalyse – Consumer Complaint Dataset

## 1. Projektziel und Datensatz
Ziel des Projekts ist es, aus einer großen Menge unstrukturierter Verbraucherbeschwerden die am häufigsten vorkommenden Themen mithilfe von NLP-Techniken zu extrahieren.

Als Datengrundlage wird der Consumer Complaint Dataset des Consumer Financial Protection Bureau (CFPB) verwendet. Der Datensatz enthält schriftliche Verbraucherbeschwerden sowie zusätzliche Metadaten und bereits vorgegebene Kategorien. Für die eigentliche Themenanalyse wird ein zufällig ausgewählter Teil des Datensatzes mit ausschließlich freien Beschwerdetexten verwendet, die vorgegebenen Kategorien werden für die Datenanalyse nicht verwendet.

## 2. Datenimport

In [ ]:
import pandas as pd
# Datensatz wurde zur weiteren Verarbeitung anfangs eingelesen.
# df = pd.read_csv(r"C:\Users\goetz\Dropbox\Studium\5. Semester\Data Analyst Projekt\dataset\complaints.csv")

# Aufruf des mittlerweile gespeicherten Datesatzes mit der StiPo der 20000 Datensätze
sample = pd.read_csv(
    r"C:\Users\goetz\Dropbox\Studium\5. Semester\Data Analyst Projekt\dataset\complaints_sample_20000.csv"
)

## 3. Explorative Datenanalyse

In [ ]:
# df.shape

(2023066, 11)

Der Datensatz enthält insgesamt 2.023.066 Einträge und 11 Merkmale. Da die Verarbeitung von über zwei Millionen Texten einen sehr hohen Rechenaufwand verursachen würde, soll für die weitere Analyse nur eine Stichprobe verwendet werden.

Wie groß diese Stichprobe sein soll, wird nach einer ersten Untersuchung der Daten entschieden. Dabei soll die Stichprobe groß genug sein, um häufig vorkommende Themen erkennen zu können, gleichzeitig aber auch mit den zur Verfügung stehenden Mitteln gut verarbeitet werden können.

In [ ]:
# df.columns

**Enthaltene Merkmale:**

"Unnamed: 0", "product_5", "narrative", "Product", "Date received", "Sub-product", "Issue", 
"Sub-issue", "Company", "State", "Timely response?"

In [ ]:
# df["narrative"].head()

Die Spalte narrative enthält die frei formulierten Beschwerdetexte und bildet damit die Grundlage für die weitere Textanalyse. Für die folgenden Schritte wird daher ausschließlich diese Textspalte betrachtet.

## 4. Auswahl der Textdaten

Da der vollständige Datensatz über zwei Millionen Beschwerdetexte enthält, wird für die weitere Analyse eine Stichprobe von 20.000 Texten verwendet. Dies entspricht ungefähr 1 % des gesamten Datensatzes. Die Stichprobe soll groß genug sein, um häufig vorkommende Themen erkennen zu können, gleichzeitig aber den Rechenaufwand bei der späteren Vektorisierung und Themenanalyse begrenzen. Die Auswahl erfolgt zufällig und mit einem festen `random_state`, damit bei einer erneuten Ausführung dieselben Datensätze ausgewählt werden.

Vor der Auswahl der Stichprobe werden leere Einträge sowie Texte mit weniger als zehn Wörtern ausgeschlossen. Sehr kurze Texte können beispielsweise nur aus einer Anrede oder wenigen allgemeinen Wörtern bestehen und enthalten damit kaum Informationen über den eigentlichen Inhalt der Beschwerde. Mit einer Mindestlänge von zehn Wörtern soll sichergestellt werden, dass die ausgewählten Texte zumindest einen gewissen inhaltlichen Umfang besitzen.

In [ ]:
CREATE_SAMPLE = False

if CREATE_SAMPLE:

    df = pd.read_csv(
        r"C:\Users\goetz\Dropbox\Studium\5. Semester\Data Analyst Projekt\dataset\complaints.csv"
    )

    valid_narratives = df[
        df["narrative"].notna() &
        (df["narrative"].str.split().str.len() >= 10)
    ]
    sample = valid_narratives.sample(
        n=20000,
        random_state=42
    )

    sample.to_csv(
        r"C:\Users\goetz\Dropbox\Studium\5. Semester\Data Analyst Projekt\dataset\complaints_sample_20000.csv",
        index=False
    )

Die ausgewählte Stichprobe mit 20.000 Datensätzen wurde als eigene CSV-Datei gespeichert. Für die weiteren Analyseschritte wird diese Datei verwendet. Dadurch muss der vollständige Datensatz mit über zwei Millionen Einträgen bei späteren Sitzungen nicht erneut verarbeitet werden. Der obige Code wurde so angepasst, dass das Notebook weiterhin vollständig über Run All ausgeführt werden kann, ohne die Stichprobe erneut zu erstellen.

## 5. Textvorverarbeitung

Vor der Vektorisierung wurde geprüft, welche Schritte der Textvorverarbeitung für die Analyse notwendig sind. Dabei wurde festgestellt, dass ein großer Teil der üblichen NLP-Vorverarbeitung bereits durch die später verwendeten Vektorisierungsverfahren übernommen werden kann.

Eine separate Tokenisierung ist nicht notwendig, da "CountVectorizer" und "TfidfVectorizer" die Texte selbst in einzelne Wörter bzw. Tokens zerlegen. Beide Verfahren können außerdem die Texte in Kleinschreibung umwandeln und Satz- und Sonderzeichen bei der Bildung der Wörter weitgehend unberücksichtigt lassen. Auch englische Stoppwörter können direkt bei der Vektorisierung entfernt werden.

Auf eine zusätzliche Lemmatisierung wird zunächst verzichtet. Nach der Vektorisierung wird geprüft, ob unterschiedliche Wortformen häufig als einzelne Merkmale auftreten und dadurch die Ergebnisse beeinflussen. Sollte dies der Fall sein, kann die Lemmatisierung anschließend ergänzt werden.

Auch Zahlen bleiben zunächst erhalten. Nach der Vektorisierung wird geprüft, ob häufig vorkommende Zahlen unter den relevanten Merkmalen auftreten und die Themenanalyse beeinflussen. Falls dies der Fall ist, können diese anschließend entfernt werden.

Die Beschwerdetexte werden daher zunächst ohne zusätzliche manuelle Bereinigung an die Vektorisierungsverfahren übergeben.

## 6. Vektorisierung

Damit die Beschwerdetexte mit mathematischen Verfahren analysiert werden können, müssen die Texte zunächst in eine numerische Form umgewandelt werden. Dafür werden zwei unterschiedliche Verfahren verwendet und anschließend miteinander verglichen.

Als erstes Verfahren wird Bag-of-Words verwendet. Dabei wird für jedes Dokument erfasst, wie häufig die einzelnen Wörter vorkommen. Als zweites Verfahren wird TF-IDF eingesetzt. Hierbei wird zusätzlich berücksichtigt, wie häufig ein Wort im gesamten Datensatz vorkommt. Wörter, die in vielen Dokumenten vorkommen, erhalten dadurch eine geringere Gewichtung.

Bei beiden Verfahren werden die Texte in Kleinschreibung umgewandelt, tokenisiert und englische Stoppwörter entfernt.

### 6.1 Bag-of-Words

Der CountVectorizer übernimmt bereits einige Schritte der Textvorverarbeitung. Die Texte werden standardmäßig in Kleinbuchstaben umgewandelt und bei der Vektorisierung in einzelne Tokens zerlegt. Satz- und Sonderzeichen werden dabei weitgehend nicht als eigene Merkmale berücksichtigt. Zusätzlich werden mit der Einstellung "stop_words='english'" häufig vorkommende englische Stoppwörter entfernt.

Weitere Bereinigungsschritte werden zunächst nicht vorgenommen. Nach der Vektorisierung wird geprüft, ob sich unter den häufigsten Merkmalen weitere inhaltlich bedeutungslose Begriffe befinden, die für die spätere Analyse entfernt werden sollten.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Bag-of-Words
bow_vectorizer = CountVectorizer(stop_words="english")

bow_matrix = bow_vectorizer.fit_transform(sample["narrative"])

In [ ]:
bow_matrix.shape

In [ ]:
# Häufigste Wörter im Bag-of-Words-Modell

import numpy as np

word_counts = np.asarray(bow_matrix.sum(axis=0)).flatten()
feature_names = bow_vectorizer.get_feature_names_out()

top_words = sorted(
    zip(feature_names, word_counts),
    key=lambda x: x[1],
    reverse=True
)[:30]

top_words

Die Betrachtung der häufigsten Merkmale zeigt, dass die Anonymisierungsplatzhalter "xxxx" und "xx" sehr häufig vorkommen und dadurch die spätere Themenanalyse beeinflussen könnten. Da diese keine inhaltliche Bedeutung besitzen, werden sie entfernt.

Außerdem treten reine Zahlen bereits unter den häufigsten Merkmalen auf. Da einzelne Zahlen für die gesuchten Themen ebenfalls keinen wesentlichen Inhalt liefern, werden auch diese aus der weiteren Analyse ausgeschlossen.

Unterschiedliche Wortformen wie "account" und "accounts" bleiben zunächst erhalten. Ob hierfür eine Lemmatisierung notwendig ist, wird im weiteren Verlauf geprüft.

In [ ]:
# Bag-of-Words - bereinigter Durchlauf

bow_vectorizer_clean = CountVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b(?!x+\b)[a-zA-Z]{2,}\b" # alle Zahlen und xx... Wörter ausschließen
)

bow_matrix_clean = bow_vectorizer_clean.fit_transform(sample["narrative"])

In [ ]:
bow_matrix_clean.shape

In [ ]:
# Häufigste Wörter nach der Bereinigung

word_counts_clean = np.asarray(bow_matrix_clean.sum(axis=0)).flatten()
feature_names_clean = bow_vectorizer_clean.get_feature_names_out()

top_words_clean = sorted(
    zip(feature_names_clean, word_counts_clean),
    key=lambda x: x[1],
    reverse=True
)[:30]

top_words_clean

Nach der Bereinigung enthalten die 20.000 Texte noch 21.184 unterschiedliche Merkmale. Die zuvor häufig vorkommenden Anonymisierungszeichen und Zahlen treten unter den häufigsten Begriffen nicht mehr auf.

### 6.2 TF-IDF
Auch der TfidfVectorizer übernimmt bei der Vektorisierung bereits verschiedene Schritte der Textvorverarbeitung. Dazu gehören die Umwandlung in Kleinbuchstaben, die Tokenisierung und die weitgehende Nichtberücksichtigung von Satz- und Sonderzeichen. Englische Stoppwörter werden über die entsprechende Einstellung ebenfalls entfernt. Für Bag-of-Words und TF-IDF werden dabei die gleichen Einstellungen zur Textvorverarbeitung verwendet, damit die Ergebnisse der beiden Vektorisierungsverfahren möglichst gut miteinander vergleichbar sind.

Die bei der vorherigen Prüfung identifizierten bedeutungslosen Platzhalter und Zahlen werden ebenfalls bei beiden Verfahren ausgeschlossen.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF - mit den gleichen Bereinigungseinstellungen wie Bag-of-Words

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b(?!x+\b)[a-zA-Z]{2,}\b"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(sample["narrative"])

In [ ]:
tfidf_matrix.shape

In [ ]:
# Wörter mit den höchsten TF-IDF-Gewichten

tfidf_scores = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

top_tfidf = sorted(
    zip(tfidf_feature_names, tfidf_scores),
    key=lambda x: x[1],
    reverse=True
)[:30]

top_tfidf

In der letzten Zeile tauchte nach dem ersten Lauf mit „xxxxxxxx“ eine X-Kombination mit mehr als vier Zeichen auf. Deshalb wurde der Code bei beiden Verfahren nochmals angepasst, sodass neben den bereits ausgeschlossenen Zahlen nun auch X-Kombinationen beliebiger Länge ausgeschlossen werden.

## 7. Semantische Analyse

### 7.1 LDA

Zur Themenextraktion wird zunächst LDA auf die bereinigte Bag-of-Words-Matrix angewendet. Als Ausgangswert werden 10 Themen gewählt. Die Anzahl dient zunächst nur als Startwert und soll anschließend anhand der Ergebnisse und des Coherence Scores überprüft werden.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# LDA zunächst mit 10 Themen
lda = LatentDirichletAllocation(
    n_components=10,
    random_state=42
)

lda.fit(bow_matrix_clean)

In [ ]:
# Wichtigste Wörter der 10 LDA-Themen

feature_names = bow_vectorizer_clean.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_words = topic.argsort()[-10:][::-1]
    words = [feature_names[i] for i in top_words]
    print(f"Thema {topic_idx + 1}: {', '.join(words)}")

Bei der Betrachtung der zehn Themen fällt auf, dass sich einige Themen inhaltlich stark überschneiden. Daher wird vermutet, dass eine geringere Anzahl an Themen zu einer klareren Trennung führen könnte. Dies soll im nächsten Schritt mithilfe des Coherence Scores überprüft werden. Da scikit-learn hierfür keine entsprechende Funktion bereitstellt, wird für die Berechnung zusätzlich die Bibliothek Gensim verwendet.

Um zu überprüfen, welche Anzahl an LDA-Themen am sinnvollsten ist, werden anschließend mehrere LDA-Modelle mit unterschiedlichen Anzahlen an Themen berechnet und mithilfe des Coherence Scores miteinander verglichen. Ein höherer c_v-Wert spricht dabei für einen stärkeren inhaltlichen Zusammenhang der Wörter innerhalb der gefundenen Themen.

Für die Berechnung wird die Bibliothek Gensim verwendet. Das bisherige LDA-Modell wurde jedoch mit scikit-learn erstellt. Gensim benötigt für die Berechnung des c_v-Scores zusätzlich die Texte in tokenisierter Form sowie ein eigenes Wörterbuch. Deshalb werden die vorhandenen Texte zunächst mit denselben Einstellungen wie beim CountVectorizer in einzelne Wörter zerlegt und daraus ein Gensim-Dictionary erstellt. Dabei werden noch keine neuen Themen berechnet oder verändert. Dieser Schritt dient lediglich dazu, die vorhandenen Daten für die anschließende Berechnung des Coherence Scores vorzubereiten.

In [ ]:
from gensim.corpora import Dictionary

# Texte mit denselben Regeln wie beim CountVectorizer tokenisieren
analyzer = bow_vectorizer_clean.build_analyzer()
tokenized_texts = [analyzer(text) for text in sample["narrative"]]

# Gensim-Wörterbuch für die Berechnung des c_v-Coherence Scores
dictionary = Dictionary(tokenized_texts)

Da sich bei zehn Topics mehrere Themen überschneiden, wird geprüft, ob eine andere Anzahl an Topics zu einer besseren thematischen Struktur führt. Dazu werden verschiedene Topic-Anzahlen berechnet und anhand des c_v-Coherence Scores miteinander verglichen.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from gensim.models import CoherenceModel

coherence_scores = []

for n_topics in range(1, 21):

    # LDA für die jeweilige Anzahl an Themen neu berechnen
    lda_test = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42
    )
    lda_test.fit(bow_matrix_clean)

    # Die 10 wichtigsten Wörter jedes Themas bestimmen
    topics = []
    for topic in lda_test.components_:
        top_indices = topic.argsort()[-10:][::-1]
        topics.append(
            [bow_vectorizer_clean.get_feature_names_out()[i]
             for i in top_indices]
        )

    # c_v-Coherence Score berechnen
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence="c_v"
    )

    score = coherence_model.get_coherence()
    coherence_scores.append(score)

    print(f"{n_topics} Themen: {score:.4f}")

Der höchste c_v-Coherence Score wird bei 12 Topics mit 0,6133 erreicht. Im nächsten Schritt wird daher geprüft, ob die zwölf Themen auch inhaltlich eine sinnvolle und klarere Trennung ergeben.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

lda_12 = LatentDirichletAllocation(
    n_components=12,
    random_state=42
)

lda_12.fit(bow_matrix_clean)

# 10 wichtigste Wörter der 12 Topics ausgeben
feature_names = bow_vectorizer_clean.get_feature_names_out()

for topic_idx, topic in enumerate(lda_12.components_):
    top_words = topic.argsort()[-10:][::-1]
    words = [feature_names[i] for i in top_words]
    print(f"Thema {topic_idx + 1}: {', '.join(words)}")

Anmerkung: Bei der Betrachtung der LDA-Ergebnisse fällt auf, dass einzelne Wörter in mehreren Topics vorkommen. Dies bedeutet jedoch nicht zwangsläufig, dass sich auch die Inhalte der Topics überschneiden, da dasselbe Wort in unterschiedlichen thematischen Zusammenhängen auftreten kann.

Thema 1: Kreditbetrug / Identitätsmissbrauch  
Thmea 2: Fehlerhafte Kreditauskünfte  
Thema 3: Medizinische Schulden und Inkasso  
Thema 4: Inkassoforderungen und Kreditauskunft  
THema 5: Zahlungsprobleme / drohende Zwangsvollstreckung  
THema 6: Streitfälle bei Kreditkartenbelastungen  
Thema 7: Scheckeinzahlungen und Verfügbarkeit von Guthaben  
Thema 8: Kontoeröffnung und -schließung / möglicher Betrug  
Thema 9: Studentendarlehen und Rückzahlung  
Thema 10: Autokredit und verspätete Zahlungen  
Thema 11: Banküberweisungen und Überweisungsdauer  
Thema 12: Hypothekenprobleme bei Wells Fargo

**Interpretation:** Die ermittelten Topics lassen sich anhand der jeweils zehn wahrscheinlichsten Wörter überwiegend gut voneinander abgrenzen und inhaltlich interpretieren. Eine deutliche inhaltliche Nähe zeigt sich jedoch zwischen Topic 5 und Topic 12. Beide beschreiben Probleme im Zusammenhang mit Hypotheken, Kreditanpassungen und Zwangsvollstreckungen. Topic 12 unterscheidet sich vor allem durch den starken Bezug zu Wells Fargo.

**Anmerkung:**  Die Benennung der Topics erfolgte anhand der jeweils zehn Wörter mit der höchsten Wahrscheinlichkeit und stellt damit eine inhaltliche Interpretation dar. Für eine weiterführende Validierung könnten zusätzlich Dokumente mit einer besonders hohen Zuordnungswahrscheinlichkeit zum jeweiligen Topic betrachtet werden. Dadurch ließe sich überprüfen, ob die gewählten Themenbezeichnungen auch durch die ursprünglichen Beschwerdetexte gestützt werden.

### 7.2 LSA

Als zweite Methode zur Ermittlung latenter Themenstrukturen wird eine Latent Semantic Analysis (LSA) durchgeführt.

Im Gegensatz zu LDA arbeitet LSA nicht mit Wahrscheinlichkeitsverteilungen. Als Grundlage wird die zuvor erstellte TF-IDF-Matrix verwendet. Mithilfe der Singulärwertzerlegung (SVD) wird diese auf eine kleinere Anzahl latenter Dimensionen reduziert. Wörter, die über die Dokumente hinweg ähnliche Strukturen aufweisen, können dadurch in gemeinsamen Dimensionen zusammengefasst werden.

Anschließend werden die Wörter mit den höchsten Gewichten je Dimension betrachtet und die gefundenen Dimensionen inhaltlich interpretiert.

Für eine direkte Vergleichbarkeit mit den zuvor ermittelten 12 LDA-Topics wird die Anzahl der latenten LSA-Dimensionen ebenfalls auf 12 festgelegt.

In [ ]:
from sklearn.decomposition import TruncatedSVD

lsa = TruncatedSVD(
    n_components=12,
    random_state=42
)

X_lsa = lsa.fit_transform(tfidf_matrix)

Um die latenten Dimensionen inhaltlich interpretieren zu können, werden für jede der 12 Dimensionen die zehn Wörter mit den höchsten positiven Gewichten ausgegeben. Diese Wörter zeigen, welche Begriffe die jeweilige Dimension besonders stark prägen.

In [ ]:
# Wörter aus dem TF-IDF-Vektorisierer
feature_names = tfidf_vectorizer.get_feature_names_out()

# 10 stärkste Wörter je LSA-Dimension
for i, component in enumerate(lsa.components_):
    top_indices = component.argsort()[-10:][::-1]
    top_words = feature_names[top_indices]
    
    print(f"Dimension {i + 1}: {', '.join(top_words)}")

Die ausgegebenen Wörter stellen jeweils die zehn höchsten positiven Gewichte einer latenten Dimension dar. Anders als bei LDA handelt es sich dabei nicht um Wahrscheinlichkeiten, sondern um Gewichte aus der SVD. Anhand dieser Wörter werden die Dimensionen im Folgenden inhaltlich interpretiert.

Dimension 1: Meldung und Bereitstellung von Kreditauskünften  
Dimension 2: Fehlerhafte Kreditauskünfte und deren Korrektur/Löschung  
Dimension 3: Verspätete Zahlungen und deren Meldung  
Dimension 4: Zahlungsprobleme bei Hypotheken / Kreditanpassungen  
Dimension 5: Scheckeinzahlungen und Verfügbarkeit von Guthaben  
Dimension 6: Kreditbetrug / Identitätsmissbrauch  
Dimension 7: Inkasso und Forderungseinzug  
Dimension 8: Kreditkartenbelastungen und Gebühren  
Dimension 9: Autokredit / Fahrzeugfinanzierung  
Dimension 10: Studentendarlehen und Schuldenerlass  
Dimension 11: Kreditanfragen und unautorisierte Kreditkartenaktivitäten  
Dimension 12: Kreditanfragen in der Kreditauskunft

## 8. Vergleich der Verfahren

![Vergleich LDA-LSA](Vergleich_LDA_LSA_v2.png)

**Ergebnisse:** Obwohl LDA und LSA mathematisch unterschiedlich arbeiten, zeigen die Ergebnisse eine überraschend große Überschneidung bei den grundlegenden Themen. Viele zentrale Themenbereiche werden von beiden Verfahren erkannt. Dazu gehören beispielsweise Kreditbetrug und Identitätsmissbrauch, fehlerhafte Kreditauskünfte, Scheckeinzahlungen, Studentendarlehen oder Autokredite.

Die Ergebnisse sind dabei nicht identisch. Teilweise fassen die Verfahren Themen unterschiedlich zusammen oder trennen sie stärker voneinander. So werden beispielsweise Zahlungs- und Hypothekenprobleme bei LDA auf mehrere Topics verteilt, während LSA diese teilweise in einer gemeinsamen Dimension zusammenfasst. LSA unterscheidet dagegen verschiedene Bereiche rund um Kreditauskünfte und Kreditanfragen stärker.

Insgesamt zeigen beide Verfahren damit trotz ihrer unterschiedlichen Vorgehensweise ein sehr ähnliches thematisches Grundgerüst des Datensatzes. Die Unterschiede liegen vor allem darin, wie die einzelnen Themen voneinander abgegrenzt bzw. zusammengefasst werden.


**Anmerkung:** Auf eine zusätzliche Lemmatisierung wurde in dieser Analyse verzichtet. In den ermittelten Topics und Dimensionen treten jedoch teilweise unterschiedliche Wortformen desselben Begriffs auf. Eine Lemmatisierung könnte diese Formen zusammenführen und dadurch die Gewichtung einzelner Merkmale verändern. Dadurch könnten gegebenenfalls weitere Begriffe unter den jeweils stärksten Wörtern erscheinen und die Interpretation einzelner Topics oder Dimensionen verbessern. Da sich die ermittelten Themen jedoch bereits überwiegend sinnvoll voneinander abgrenzen und interpretieren lassen, wurde auf diesen zusätzlichen Verarbeitungsschritt für die vorliegende Analyse verzichtet.